In [1]:
import numpy as np
import spacy
import warnings
import os
import time
import pandas as pd
from dotenv import load_dotenv
from scipy.optimize import minimize
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import logging

# --- Together AI Client ---
from together import Together

# Suppress warnings for clean terminal output
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# --- Qiskit Imports ---
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import Sampler as LocalSampler

# --- Configuration ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Define the old CSV to read from and the new CSV to write to
OLD_CSV_FILENAME = "qrag_telemetry_N150_run_1783611471_final.csv"  # <-- UPDATE THIS TO YOUR PREVIOUS RUN'S CSV
RUN_TIMESTAMP = int(time.time())
CSV_FILENAME = f"qrag_telemetry_Updated_run_{RUN_TIMESTAMP}.csv"

# Load environment variables
load_dotenv()

# ==============================================================================
# DATASET PLACEHOLDER (NEW AGENT-PATIENT INVERSION SENTENCES)
# ==============================================================================
NEW_DATABASE = [
  {
    "class": "Agent-Patient Inversion",
    "text": "The shaved ice scraped the heavy ice shaver.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shaved ice",
    "conflict": "the heavy ice shaver"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The crushed garlic squeezed the heavy garlic press.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the crushed garlic",
    "conflict": "the heavy garlic press"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed vegetables boiled the bamboo vegetable steamer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed vegetables",
    "conflict": "the bamboo vegetable steamer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The brewed espresso extracted the heavy espresso machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the brewed espresso",
    "conflict": "the heavy espresso machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked nuts snapped the heavy metal nutcracker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked nuts",
    "conflict": "the heavy metal nutcracker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed garment smoothed the handheld garment steamer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed garment",
    "conflict": "the handheld garment steamer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The painted wall rolled the fuzzy paint roller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the painted wall",
    "conflict": "the fuzzy paint roller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut glass scored the diamond-tipped glass cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut glass",
    "conflict": "the diamond-tipped glass cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted solder wicked the copper desoldering braid.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted solder",
    "conflict": "the copper desoldering braid"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The polished metal buffed the cotton buffing wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the polished metal",
    "conflict": "the cotton buffing wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut tile snapped the manual tile cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut tile",
    "conflict": "the manual tile cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The poured driveway floated the magnesium bull float.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the poured driveway",
    "conflict": "the magnesium bull float"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The smoothed plaster troweled the flat masonry trowel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the smoothed plaster",
    "conflict": "the flat masonry trowel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The secured drywall screwed the electric drywall gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the secured drywall",
    "conflict": "the electric drywall gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut cardboard scissored the sharp craft scissors.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut cardboard",
    "conflict": "the sharp craft scissors"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cleaned monitor wiped the disposable screen wipe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cleaned monitor",
    "conflict": "the disposable screen wipe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The incubated cells warmed the regulated cell incubator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the incubated cells",
    "conflict": "the regulated cell incubator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The trimmed hedge sheared the electric hedge trimmer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the trimmed hedge",
    "conflict": "the electric hedge trimmer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The whisked cream beat the stainless wire balloon whisk.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the whisked cream",
    "conflict": "the stainless wire balloon whisk"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The strained broth drained the fine mesh conical strainer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the strained broth",
    "conflict": "the fine mesh conical strainer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sifted powdered sugar filtered the mechanical flour sifter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sifted powdered sugar",
    "conflict": "the mechanical flour sifter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened tin punctured the heavy-duty manual can opener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened tin",
    "conflict": "the heavy-duty manual can opener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The uncorked champagne pulled the articulated sommelier corkscrew.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the uncorked champagne",
    "conflict": "the articulated sommelier corkscrew"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baked sourdough heated the enameled cast-iron dutch oven.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baked sourdough",
    "conflict": "the enameled cast-iron dutch oven"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The toasted bagel browned the four-slot popup toaster.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the toasted bagel",
    "conflict": "the four-slot popup toaster"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mixed dough stirred the heavy planetary stand mixer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mixed dough",
    "conflict": "the heavy planetary stand mixer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ground peppercorns crushed the ceramic burr pepper grinder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ground peppercorns",
    "conflict": "the ceramic burr pepper grinder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The juiced lemons squeezed the motorized citrus juicer press.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the juiced lemons",
    "conflict": "the motorized citrus juicer press"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chopped onions diced the commercial food processor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chopped onions",
    "conflict": "the commercial food processor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The flipped burger tossed the slotted stainless steel spatula.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the flipped burger",
    "conflict": "the slotted stainless steel spatula"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scooped sorbet dug the aluminum anti-freeze ice cream scoop.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scooped sorbet",
    "conflict": "the aluminum anti-freeze ice cream scoop"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled fondant flattened the silicone non-stick rolling pin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled fondant",
    "conflict": "the silicone non-stick rolling pin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The grated parmesan shredded the stainless steel rotary cheese grater.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the grated parmesan",
    "conflict": "the stainless steel rotary cheese grater"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The peeled apples skinned the swivel-blade vegetable peeler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the peeled apples",
    "conflict": "the swivel-blade vegetable peeler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The carved roast sliced the cordless electric carving knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the carved roast",
    "conflict": "the cordless electric carving knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The brewed peppermint steeped the stainless steel tea infuser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the brewed peppermint",
    "conflict": "the stainless steel tea infuser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted chocolate heated the copper double boiler pot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted chocolate",
    "conflict": "the copper double boiler pot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tenderized cutlet pounded the dual-sided meat mallet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tenderized cutlet",
    "conflict": "the dual-sided meat mallet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spiralized squash twisted the countertop vegetable spiralizer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spiralized squash",
    "conflict": "the countertop vegetable spiralizer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The microplaned nutmeg zested the razor-sharp microplane grater.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the microplaned nutmeg",
    "conflict": "the razor-sharp microplane grater"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cored pineapple pierced the cylindrical stainless apple corer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cored pineapple",
    "conflict": "the cylindrical stainless apple corer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pitted olives punched the handheld spring-loaded cherry pitter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pitted olives",
    "conflict": "the handheld spring-loaded cherry pitter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hulled berries plucked the clawed strawberry huller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hulled berries",
    "conflict": "the clawed strawberry huller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The portioned dough dropped the ratcheting cookie scoop.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the portioned dough",
    "conflict": "the ratcheting cookie scoop"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The whipped meringue aerated the nitrous oxide cream dispenser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the whipped meringue",
    "conflict": "the nitrous oxide cream dispenser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The boiled pasta timed the magnetic digital kitchen timer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the boiled pasta",
    "conflict": "the magnetic digital kitchen timer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The poached salmon shaped the perforated silicone egg poacher.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the poached salmon",
    "conflict": "the perforated silicone egg poacher"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The wok-fried rice tossed the seasoned carbon steel wok.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the wok-fried rice",
    "conflict": "the seasoned carbon steel wok"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The deep-fried tempura submerged the stainless wire deep fryer basket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the deep-fried tempura",
    "conflict": "the stainless wire deep fryer basket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The seared tuna blackened the seasoned cast iron skillet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the seared tuna",
    "conflict": "the seasoned cast iron skillet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The slow-cooked chili simmered the programmable electric slow cooker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the slow-cooked chili",
    "conflict": "the programmable electric slow cooker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pressure-cooked stew contained the high-pressure stovetop cooker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pressure-cooked stew",
    "conflict": "the high-pressure stovetop cooker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The air-fried fries crisped the convection digital air fryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the air-fried fries",
    "conflict": "the convection digital air fryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The smoked salmon flavored the horizontal offset wood smoker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the smoked salmon",
    "conflict": "the horizontal offset wood smoker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The grilled sausages seared the propane outdoor gas grill.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the grilled sausages",
    "conflict": "the propane outdoor gas grill"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled champagne cooled the thermoelectric wine cooler fridge.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled champagne",
    "conflict": "the thermoelectric wine cooler fridge"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen sorbet molded the flexible silicone popsicle molds.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen sorbet",
    "conflict": "the flexible silicone popsicle molds"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The decanted merlot aerated the wide-based crystal wine decanter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the decanted merlot",
    "conflict": "the wide-based crystal wine decanter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sawed lumber cut the compound sliding miter saw.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sawed lumber",
    "conflict": "the compound sliding miter saw"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sanded flooring smoothed the heavy orbital floor sander.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sanded flooring",
    "conflict": "the heavy orbital floor sander"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The glued veneer bonded the industrial polyurethane wood glue.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the glued veneer",
    "conflict": "the industrial polyurethane wood glue"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clamped assembly squeezed the heavy-duty forged C-clamp.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clamped assembly",
    "conflict": "the heavy-duty forged C-clamp"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The soldered wires melted the fine-tipped butane soldering iron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the soldered wires",
    "conflict": "the fine-tipped butane soldering iron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured dimension extended the rigid fiberglass tape measure.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured dimension",
    "conflict": "the rigid fiberglass tape measure"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The routed edge carved the variable-speed plunge wood router.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the routed edge",
    "conflict": "the variable-speed plunge wood router"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The planed surface shaved the cast-iron low-angle block plane.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the planed surface",
    "conflict": "the cast-iron low-angle block plane"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bolted chassis torqued the pneumatic impact socket wrench.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bolted chassis",
    "conflict": "the pneumatic impact socket wrench"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The welded framework fused the oxyacetylene gas welding torch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the welded framework",
    "conflict": "the oxyacetylene gas welding torch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chiseled block broke the hardened steel masonry cold chisel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chiseled block",
    "conflict": "the hardened steel masonry cold chisel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The filed steel scraped the double-cut coarse bastard file.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the filed steel",
    "conflict": "the double-cut coarse bastard file"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drilled stud penetrated the brushless cordless power drill.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drilled stud",
    "conflict": "the brushless cordless power drill"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The leveled framing balanced the anodized aluminum bubble level.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the leveled framing",
    "conflict": "the anodized aluminum bubble level"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The marked foundation lined the powdered blue chalk line.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the marked foundation",
    "conflict": "the powdered blue chalk line"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The gripped fitting grabbed the forged steel adjustable pipe wrench.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the gripped fitting",
    "conflict": "the forged steel adjustable pipe wrench"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut cable snipped the high-leverage diagonal wire cutters.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut cable",
    "conflict": "the high-leverage diagonal wire cutters"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stripped insulation exposed the precision manual wire strippers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stripped insulation",
    "conflict": "the precision manual wire strippers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The crimped lug squeezed the hydraulic metal wire crimping tool.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the crimped lug",
    "conflict": "the hydraulic metal wire crimping tool"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stapled vapor barrier punched the heavy-duty manual staple gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stapled vapor barrier",
    "conflict": "the heavy-duty manual staple gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The riveted flashing popped the long-handle manual pop riveter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the riveted flashing",
    "conflict": "the long-handle manual pop riveter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The painted molding brushed the synthetic angled sash brush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the painted molding",
    "conflict": "the synthetic angled sash brush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scraped wallpaper peeled the stiff high-carbon putty knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scraped wallpaper",
    "conflict": "the stiff high-carbon putty knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caulked baseboard squeezed the thrust-ratio dripless caulk gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caulked baseboard",
    "conflict": "the thrust-ratio dripless caulk gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tightened fastener turned the chrome crescent adjustable wrench.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tightened fastener",
    "conflict": "the chrome crescent adjustable wrench"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The loosened bracket grabbed the forged locking vise-grip pliers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the loosened bracket",
    "conflict": "the forged locking vise-grip pliers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pryed floorboard lifted the hexagonal steel crowbar pry bar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pryed floorboard",
    "conflict": "the hexagonal steel crowbar pry bar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The insulated conduit wrapped the weather-resistant black electrical tape.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the insulated conduit",
    "conflict": "the weather-resistant black electrical tape"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The lubricated gear sprayed the aerosol synthetic penetrating oil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the lubricated gear",
    "conflict": "the aerosol synthetic penetrating oil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sharpened chisel ground the variable-speed heavy bench grinder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sharpened chisel",
    "conflict": "the variable-speed heavy bench grinder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The engraved serial scratched the high-speed rotary engraving tool.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the engraved serial",
    "conflict": "the high-speed rotary engraving tool"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mixed mortar churned the towable portable cement mixer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mixed mortar",
    "conflict": "the towable portable cement mixer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The framed partition shot the sequential pneumatic framing nailer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the framed partition",
    "conflict": "the sequential pneumatic framing nailer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The taped joint covered the self-adhesive fiberglass drywall tape.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the taped joint",
    "conflict": "the self-adhesive fiberglass drywall tape"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mudded corner spread the flexible blue-steel taping knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mudded corner",
    "conflict": "the flexible blue-steel taping knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hoisted block lifted the rolling hydraulic engine shop hoist.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hoisted block",
    "conflict": "the rolling hydraulic engine shop hoist"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The supported axle jacked the heavy-duty ratcheting steel jack stands.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the supported axle",
    "conflict": "the heavy-duty ratcheting steel jack stands"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The inflated inner tube pumped the twin-cylinder stationary air compressor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the inflated inner tube",
    "conflict": "the twin-cylinder stationary air compressor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The greased ball joint injected the lever-action heavy grease gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the greased ball joint",
    "conflict": "the lever-action heavy grease gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The jump-started pickup clamped the heavy-gauge copper jumper cables.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the jump-started pickup",
    "conflict": "the heavy-gauge copper jumper cables"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The injected anesthetic pierced the sterile stainless hypodermic needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the injected anesthetic",
    "conflict": "the sterile stainless hypodermic needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drawn plasma vacuumed the single-use plastic blood syringe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drawn plasma",
    "conflict": "the single-use plastic blood syringe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stitched incision threaded the absorbable curved surgical suture needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stitched incision",
    "conflict": "the absorbable curved surgical suture needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bandaged burn wrapped the non-adherent sterile cotton gauze roll.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bandaged burn",
    "conflict": "the non-adherent sterile cotton gauze roll"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The listened murmur thumped the dual-lumen cardiology acoustic stethoscope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the listened murmur",
    "conflict": "the dual-lumen cardiology acoustic stethoscope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured basal temperature beeped the flexible digital oral thermometer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured basal temperature",
    "conflict": "the flexible digital oral thermometer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The weighed infant balanced the calibrated digital pediatric medical scale.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the weighed infant",
    "conflict": "the calibrated digital pediatric medical scale"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun urinalysis centrifuged the variable-speed electric laboratory centrifuge.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun urinalysis",
    "conflict": "the variable-speed electric laboratory centrifuge"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pipetted antibodies transferred the calibrated adjustable volume micropipette.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pipetted antibodies",
    "conflict": "the calibrated adjustable volume micropipette"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The magnified erythrocytes focused the binocular optical compound microscope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the magnified erythrocytes",
    "conflict": "the binocular optical compound microscope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The diagnosed fracture x-rayed the flat-panel digital radiography scanner.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the diagnosed fracture",
    "conflict": "the flat-panel digital radiography scanner"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scanned cortex imaged the superconducting loud MRI machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scanned cortex",
    "conflict": "the superconducting loud MRI machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The monitored saturation beeped the infrared optical fingertip pulse oximeter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the monitored saturation",
    "conflict": "the infrared optical fingertip pulse oximeter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The checked hypertension pumped the manual inflatable aneroid sphygmomanometer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the checked hypertension",
    "conflict": "the manual inflatable aneroid sphygmomanometer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stabilized vertebrae supported the molded rigid foam cervical collar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stabilized vertebrae",
    "conflict": "the molded rigid foam cervical collar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The braced ligament hinged the articulated flexible neoprene knee brace.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the braced ligament",
    "conflict": "the articulated flexible neoprene knee brace"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The casted fibula hardened the water-activated synthetic fiberglass cast.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the casted fibula",
    "conflict": "the water-activated synthetic fiberglass cast"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The supported gait leaned the adjustable aluminum underarm medical crutches.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the supported gait",
    "conflict": "the adjustable aluminum underarm medical crutches"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pushed amputee rolled the lightweight folding manual transit wheelchair.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pushed amputee",
    "conflict": "the lightweight folding manual transit wheelchair"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cleared mucous suctioned the portable electric medical vacuum aspirator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cleared mucous",
    "conflict": "the portable electric medical vacuum aspirator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The oxygenated preemie breathed the soft transparent silicone nasal cannula.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the oxygenated preemie",
    "conflict": "the soft transparent silicone nasal cannula"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ventilated patient pumped the microprocessor-controlled mechanical hospital ventilator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ventilated patient",
    "conflict": "the microprocessor-controlled mechanical hospital ventilator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shocked arrhythmia defibrillated the portable automated external defibrillator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shocked arrhythmia",
    "conflict": "the portable automated external defibrillator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clamped vein locked the titanium surgical Kelly hemostat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clamped vein",
    "conflict": "the titanium surgical Kelly hemostat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cauterized bleeding burned the disposable handheld electrocautery pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cauterized bleeding",
    "conflict": "the disposable handheld electrocautery pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The intubated larynx guided the wire-reinforced flexible endotracheal tube.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the intubated larynx",
    "conflict": "the wire-reinforced flexible endotracheal tube"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drained abscess flowed the sterile silicone surgical Foley catheter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drained abscess",
    "conflict": "the sterile silicone surgical Foley catheter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut fascia snipped the blunt-tipped curved Mayo surgical scissors.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut fascia",
    "conflict": "the blunt-tipped curved Mayo surgical scissors"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The retracted muscle pulled the self-retaining metal surgical retractor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the retracted muscle",
    "conflict": "the self-retaining metal surgical retractor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated abdomen shone the multi-bulb overhead surgical shadowless light.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated abdomen",
    "conflict": "the multi-bulb overhead surgical shadowless light"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stapled manuscript punched the spring-powered desktop metal stapler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stapled manuscript",
    "conflict": "the spring-powered desktop metal stapler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hole-punched binder pressed the adjustable mechanical three-hole punch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hole-punched binder",
    "conflict": "the adjustable mechanical three-hole punch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The erased typo rubbed the kneadable pink art rubber eraser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the erased typo",
    "conflict": "the kneadable pink art rubber eraser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The highlighted passage colored the fluorescent neon yellow text highlighter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the highlighted passage",
    "conflict": "the fluorescent neon yellow text highlighter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The glued cardboard adhered the washable non-toxic sticky glue stick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the glued cardboard",
    "conflict": "the washable non-toxic sticky glue stick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stamped invoice inked the customized self-inking corporate rubber stamp.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stamped invoice",
    "conflict": "the customized self-inking corporate rubber stamp"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shredded statement sliced the micro-cut noisy office paper shredder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shredded statement",
    "conflict": "the micro-cut noisy office paper shredder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The taped parcel sealed the ergonomic handheld packaging clear tape dispenser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the taped parcel",
    "conflict": "the ergonomic handheld packaging clear tape dispenser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clipped dossier pinched the folded black steel spring binder clip.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clipped dossier",
    "conflict": "the folded black steel spring binder clip"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pinned memo stabbed the colorful sharp plastic bulletin pushpin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pinned memo",
    "conflict": "the colorful sharp plastic bulletin pushpin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bound portfolio clamped the manual plastic comb document binding machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bound portfolio",
    "conflict": "the manual plastic comb document binding machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The laminated menu melted the four-roller thermal document pouch laminator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the laminated menu",
    "conflict": "the four-roller thermal document pouch laminator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The printed spreadsheet inked the high-capacity desktop color inkjet printer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the printed spreadsheet",
    "conflict": "the high-capacity desktop color inkjet printer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scanned contract digitized the duplex automatic sheet document feeder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scanned contract",
    "conflict": "the duplex automatic sheet document feeder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The faxed authorization transmitted the thermal analog dial-up fax machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the faxed authorization",
    "conflict": "the thermal analog dial-up fax machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The projected slideshow beamed the high-lumen digital classroom DLP projector.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the projected slideshow",
    "conflict": "the high-lumen digital classroom DLP projector"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The calculated total tallied the ten-digit solar desktop financial calculator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the calculated total",
    "conflict": "the ten-digit solar desktop financial calculator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The weighed parcel balanced the high-precision digital shipping postage scale.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the weighed parcel",
    "conflict": "the high-precision digital shipping postage scale"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The metered envelope stamped the network-connected electronic automated postage meter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the metered envelope",
    "conflict": "the network-connected electronic automated postage meter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The typed essay clicked the tactile mechanical backlit computer keyboard.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the typed essay",
    "conflict": "the tactile mechanical backlit computer keyboard"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clicked hyperlink tracked the ergonomic wireless laser optical mouse.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clicked hyperlink",
    "conflict": "the ergonomic wireless laser optical mouse"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tapped application registered the pressure-sensitive active digital stylus pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tapped application",
    "conflict": "the pressure-sensitive active digital stylus pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The written signature scribbled the fine-point blue retractable ballpoint pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the written signature",
    "conflict": "the fine-point blue retractable ballpoint pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drafted schematic marked the lead-feeding mechanical architectural drafting pencil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drafted schematic",
    "conflict": "the lead-feeding mechanical architectural drafting pencil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drawn arc rotated the adjustable steel geometric drafting compass.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drawn arc",
    "conflict": "the adjustable steel geometric drafting compass"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured margin guided the transparent clear plastic acrylic ruler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured margin",
    "conflict": "the transparent clear plastic acrylic ruler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The read paperback flipped the yellowed printed paper novel pages.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the read paperback",
    "conflict": "the yellowed printed paper novel pages"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bookmarked chapter clipped the decorative magnetic folded page marker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bookmarked chapter",
    "conflict": "the decorative magnetic folded page marker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The organized receipts sorted the multi-pocket expanding accordion file folder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the organized receipts",
    "conflict": "the multi-pocket expanding accordion file folder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The labeled bin printed the portable thermal Bluetooth adhesive label maker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the labeled bin",
    "conflict": "the portable thermal Bluetooth adhesive label maker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The watered seedlings showered the galvanized steel long-spout watering can.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the watered seedlings",
    "conflict": "the galvanized steel long-spout watering can"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pruned rosebush snipped the titanium bypass garden pruning shears.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pruned rosebush",
    "conflict": "the titanium bypass garden pruning shears"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dug trench lifted the tempered pointed steel digging shovel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dug trench",
    "conflict": "the tempered pointed steel digging shovel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chopped kindling split the forged heavy carbon splitting maul.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chopped kindling",
    "conflict": "the forged heavy carbon splitting maul"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caught trout hooked the treble-barbed metallic fishing spinner lure.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caught trout",
    "conflict": "the treble-barbed metallic fishing spinner lure"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The netted moth trapped the long-handled fine mesh entomology butterfly net.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the netted moth",
    "conflict": "the long-handled fine mesh entomology butterfly net"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The braked sedan clamped the semi-metallic ventilated ceramic disc brake pads.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the braked sedan",
    "conflict": "the semi-metallic ventilated ceramic disc brake pads"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steered galleon rotated the brass-bound wooden maritime ship wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steered galleon",
    "conflict": "the brass-bound wooden maritime ship wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rowed skiff paddled the varnished long wooden boat oars.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rowed skiff",
    "conflict": "the varnished long wooden boat oars"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sailed dinghy caught the wind-filled triangular canvas jib sail.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sailed dinghy",
    "conflict": "the wind-filled triangular canvas jib sail"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pedaled tandem turned the spiked aluminum mountain bicycle pedals.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pedaled tandem",
    "conflict": "the spiked aluminum mountain bicycle pedals"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The anchored schooner dropped the galvanized heavy iron mushroom anchor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the anchored schooner",
    "conflict": "the galvanized heavy iron mushroom anchor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The accelerated chopper twisted the textured rubber motorcycle throttle grip.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the accelerated chopper",
    "conflict": "the textured rubber motorcycle throttle grip"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shifted overdrive moved the leather-wrapped manual car transmission stick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shifted overdrive",
    "conflict": "the leather-wrapped manual car transmission stick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The wiped drizzle cleared the articulated rubber automotive windshield wiper blades.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the wiped drizzle",
    "conflict": "the articulated rubber automotive windshield wiper blades"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The honked warning sounded the central steering wheel airbag horn button.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the honked warning",
    "conflict": "the central steering wheel airbag horn button"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The fueled tractor pumped the high-flow heavy unleaded gasoline nozzle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the fueled tractor",
    "conflict": "the high-flow heavy unleaded gasoline nozzle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charged tesla plugged the thick high-voltage supercharger cable.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charged tesla",
    "conflict": "the thick high-voltage supercharger cable"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The locked hatchback clicked the remote keyless entry fob transmitter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the locked hatchback",
    "conflict": "the remote keyless entry fob transmitter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The navigated highway displayed the suction-mounted dashboard satellite GPS unit.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the navigated highway",
    "conflict": "the suction-mounted dashboard satellite GPS unit"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hit softball swung the composite aluminum slow-pitch bat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hit softball",
    "conflict": "the composite aluminum slow-pitch bat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The kicked free-kick booted the spiked leather firm-ground soccer cleat.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the kicked free-kick",
    "conflict": "the spiked leather firm-ground soccer cleat"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caught pop-fly squeezed the webbed oversized leather outfielder glove.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caught pop-fly",
    "conflict": "the webbed oversized leather outfielder glove"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shot slapshot slapped the flexible carbon fiber composite hockey stick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shot slapshot",
    "conflict": "the flexible carbon fiber composite hockey stick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spiked serve slapped the tensioned nylon indoor volleyball net.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spiked serve",
    "conflict": "the tensioned nylon indoor volleyball net"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed laundry tumbled the high-efficiency front-loading washing machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed laundry",
    "conflict": "the high-efficiency front-loading washing machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried towels spun the electric vented drum clothes dryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried towels",
    "conflict": "the electric vented drum clothes dryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ironed blouse pressed the ceramic-coated hot steam iron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ironed blouse",
    "conflict": "the ceramic-coated hot steam iron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mopped linoleum wrung the absorbent microfiber spin floor mop.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mopped linoleum",
    "conflict": "the absorbent microfiber spin floor mop"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The swept patio brushed the wide synthetic-bristled outdoor push broom.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the swept patio",
    "conflict": "the wide synthetic-bristled outdoor push broom"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dusted baseboards wiped the extendable ostrich electrostatic feather duster.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dusted baseboards",
    "conflict": "the extendable ostrich electrostatic feather duster"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The disinfected toilet sprayed the chlorine-based antibacterial cleaning spray.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the disinfected toilet",
    "conflict": "the chlorine-based antibacterial cleaning spray"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The purified allergens filtered the activated-carbon true HEPA room purifier.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the purified allergens",
    "conflict": "the activated-carbon true HEPA room purifier"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The humidified nursery misted the quiet ultrasonic cool mist room humidifier.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the humidified nursery",
    "conflict": "the quiet ultrasonic cool mist room humidifier"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dehumidified cellar dried the continuous-drain heavy compressor dehumidifier.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dehumidified cellar",
    "conflict": "the continuous-drain heavy compressor dehumidifier"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The circulated breeze blew the rotating oscillating pedestal standing fan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the circulated breeze",
    "conflict": "the rotating oscillating pedestal standing fan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The exhausted grease vented the stainless ducted overhead range hood.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the exhausted grease",
    "conflict": "the stainless ducted overhead range hood"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated stairwell lit the dimmable warm-white LED light bulb.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated stairwell",
    "conflict": "the dimmable warm-white LED light bulb"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The darkened theater blocked the thick thermal room blackout curtains.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the darkened theater",
    "conflict": "the thick thermal room blackout curtains"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The secured entryway locked the reinforced solid brass keyed deadbolt.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the secured entryway",
    "conflict": "the reinforced solid brass keyed deadbolt"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The silenced chime pressed the tactile digital alarm snooze button.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the silenced chime",
    "conflict": "the tactile digital alarm snooze button"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The recorded broadcast programmed the hard-drive based digital video recorder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the recorded broadcast",
    "conflict": "the hard-drive based digital video recorder"
  }
]

# ==============================================================================
# DATASET CALIBRATION (REDUCING STRAWMAN GRADIENTS)
# ==============================================================================
def smooth_syntactic_gradients(db):
    for i, item in enumerate(db):
        if i % 3 == 0:
            # Boost Agentic (BGE): Add query keywords to truth to artificially raise Cross-Encoder score
            base_truth = item['truth'].replace(".", "")
            item['truth'] = f"{base_truth} is the target for: {item['query'].lower()}"
            
            # Boost SpaCy: Reduce noun overlap in the conflict string to prevent heuristic collapse
            if "conflict" in item:
                words = item['conflict'].split()
                if len(words) > 1:
                    item['conflict'] = words[-1] + "."
    return db

# Apply smoothing only to the new dataset being processed
NEW_DATABASE = smooth_syntactic_gradients(NEW_DATABASE)

# ==============================================================================
# PART 2: THE ALGORITHMIC PARSERS
# ==============================================================================

class SpacyParser:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        
    def parse(self, sentence, truth, conflict):
        doc = self.nlp(sentence)
        extracted_core = []
        for token in doc:
            if token.dep_ in ("nsubj", "ROOT", "dobj", "pobj"):
                extracted_core.append(token.lemma_.lower())
                
        core_str = " ".join(extracted_core)
        truth_overlap = sum(1 for word in core_str.split() if word in truth.lower())
        conflict_overlap = sum(1 for word in core_str.split() if word in conflict.lower())
        return 1 if truth_overlap >= conflict_overlap else 0

class AgenticParser:
    def __init__(self):
        print("Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...")
        self.reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

    def parse(self, query, truth, conflict):
        scores = self.reranker.predict([(query, truth), (query, conflict)])
        return 1 if scores[0] > scores[1] else 0

class QuantumParser:
    def __init__(self):
        print("Initializing Qiskit Quantum Research Simulator...")
        self.sampler = LocalSampler()
        self.nlp = spacy.load("en_core_web_sm")
        self.shots = 4096  
        self.trained_models = {}

    def _parse_to_circuit(self, doc):
        tokens = [t for t in doc if t.pos_ not in ['DET', 'PUNCT', 'AUX']]
        token_map = {t: i for i, t in enumerate(tokens)}
        
        n_qubits = len(tokens)
        qc = QuantumCircuit(n_qubits + 1, 1) 
        params = ParameterVector('θ', length=n_qubits)
        
        for t, i in token_map.items(): 
            qc.ry(params[i], i)
            
        for t, i in token_map.items():
            if t.head in token_map and t.head != t:
                qc.cz(i, token_map[t.head]) 
                
        for i in range(n_qubits):
            qc.cx(i, n_qubits)
            
        qc.measure(n_qubits, 0)
        return qc, params

    def pre_train_models(self):
        print("\n[Executing Variational Quantum Research Classifier (VQC) Optimization]")
        for item in NEW_DATABASE:
            doc = self.nlp(item['text'])
            circuit, params = self._parse_to_circuit(doc)
            
            def objective_function(param_values):
                job = self.sampler.run([circuit], parameter_values=[param_values], shots=self.shots)
                quasi_dists = job.result().quasi_dists[0]
                prob_0 = quasi_dists.get(0, 0.0)
                return -prob_0 

            initial_params = np.random.rand(len(params)) * np.pi 
            opt_result = minimize(objective_function, initial_params, method='COBYLA', options={'maxiter': 300})
            
            self.trained_models[item['text']] = {
                'circuit': circuit, 
                'trained_params': opt_result.x
            }

    def parse(self, sentence):
        if sentence not in self.trained_models: return 0
        model = self.trained_models[sentence]
        job = self.sampler.run([model['circuit']], parameter_values=[model['trained_params']], shots=self.shots)
        quasi_dists = job.result().quasi_dists[0]
        prob_0 = quasi_dists.get(0, 0.0)
        return 1 if prob_0 > 0.5 else 0

# ==============================================================================
# PART 3: GENERATION & RAGAS METRICS
# ==============================================================================

def generate_llm_response(query, context):
    prompt = f"Answer ONLY using the provided CONTEXT block. Do not use outside knowledge. 1 sentence max.\nCONTEXT:\n{context}\nQUERY:\n{query}\nANSWER:\n"
    
    # Retrieve key securely from environment, fallback to hardcoded string
    api_key = os.environ.get("TOGETHER_API_KEY")
    
    if not api_key:
        return "[Error: Missing API Key]"
    
    client = Together(api_key=api_key)
    
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct-Lite",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0.1
        )
        ans = response.choices[0].message.content.strip().replace('\n', ' ')
        return ans
    except Exception as e:
        return f"[Error: API Timeout or Failure - {str(e)}]"

class RAGMetrics:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def calculate_ragas(self, query, context, answer):
        q_emb = self.model.encode(query)
        c_emb = self.model.encode(context)
        a_emb = self.model.encode(answer)
        
        ctx_rel = max(0.0, float(cosine_similarity([q_emb], [c_emb])[0][0] * 100))
        faith = max(0.0, float(cosine_similarity([c_emb], [a_emb])[0][0] * 100))
        ans_rel = max(0.0, float(cosine_similarity([q_emb], [a_emb])[0][0] * 100))
        
        return ctx_rel, faith, ans_rel

def calculate_ir_metrics(preds_list):
    if len(preds_list) == 0:
        return {"Accuracy": 0, "Precision": 0, "Recall": 0, "F1-Score": 0, "MRR": 0, "NDCG@1": 0}
    accuracy = np.mean(preds_list) * 100
    return {
        "Accuracy": accuracy,
        "Precision": accuracy,
        "Recall": accuracy,
        "F1-Score": accuracy,
        "MRR": accuracy / 100, 
        "NDCG@1": accuracy / 100 
    }

# ==============================================================================
# PART 4: CSV LOGGING ENGINE & MERGE LOGIC
# ==============================================================================

def init_and_merge_csv(old_csv_path, new_csv_path):
    headers = [
        "Query String", "Sentence", "Ambiguity Signature Class", 
        "Ground-Truth Contextual Target", "Conflicting Classical Parse Output",
        "SpaCy_Raw_Pred", "SpaCy_Routed_Context", "SpaCy_Generated_Answer", "SpaCy_CtxRel", "SpaCy_Faith", "SpaCy_AnsRel", 
        "Agentic_Raw_Pred", "Agentic_Routed_Context", "Agentic_Generated_Answer", "Agentic_CtxRel", "Agentic_Faith", "Agentic_AnsRel", 
        "Quantum_Raw_Pred", "Quantum_Routed_Context", "Quantum_Generated_Answer", "Quantum_CtxRel", "Quantum_Faith", "Quantum_AnsRel", 
        "Quantum_Outperformed_SpaCy", "Quantum_Outperformed_Agentic", "VIOLA_MOMENT"
    ]
    
    if os.path.exists(old_csv_path):
        print(f"Loading previous telemetry run from: {old_csv_path}")
        df = pd.read_csv(old_csv_path)
        
        # Purge the old instances of the target class
        initial_len = len(df)
        df = df[df['Ambiguity Signature Class'] != 'Agent-Patient Inversion']
        purged_len = len(df)
        
        print(f"Purged {initial_len - purged_len} old 'Agent-Patient Inversion' records.")
        df.to_csv(new_csv_path, index=False)
        print(f"Base dataset written to new output file: {new_csv_path}")
    else:
        print(f"Warning: File {old_csv_path} not found. Starting a fresh telemetry run.")
        df = pd.DataFrame(columns=headers)
        df.to_csv(new_csv_path, index=False)

def log_experiment(row_data):
    df = pd.DataFrame([row_data])
    df.to_csv(CSV_FILENAME, mode='a', header=False, index=False)

# ==============================================================================
# PART 5: MAIN EXECUTION
# ==============================================================================

if __name__ == '__main__':
    print(f"[{time.strftime('%H:%M:%S')}] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW={len(NEW_DATABASE)})")
    
    # 1. Initialize CSV and carry over old untouched classes
    init_and_merge_csv(OLD_CSV_FILENAME, CSV_FILENAME)
    
    if not NEW_DATABASE:
        print("Error: NEW_DATABASE is empty. Please populate it with the new JSON data and run again.")
        exit()

    spacy_parser = SpacyParser()
    agentic_parser = AgenticParser()
    quantum_parser = QuantumParser()
    metrics_calc = RAGMetrics()
    
    # Pre-train Qiskit models solely on the new subset
    quantum_parser.pre_train_models()

    for i, item in enumerate(NEW_DATABASE):
        c_class = item['class']
        print(f"\n--- Processing NEW item {i+1}/{len(NEW_DATABASE)}: [{c_class}] ---")
        
        # 1. Routing Predictions
        spacy_pred = spacy_parser.parse(item['text'], item['truth'], item['conflict'])
        agentic_pred = agentic_parser.parse(item['query'], item['truth'], item['conflict'])
        quantum_pred = quantum_parser.parse(item['text'])
        
        # 2. Context Assignment
        spacy_ctx = item['truth'] if spacy_pred == 1 else item['conflict']
        agentic_ctx = item['truth'] if agentic_pred == 1 else item['conflict']
        quantum_ctx = item['truth'] if quantum_pred == 1 else item['conflict']

        # 3. LLM Answer Generation 
        spacy_ans = generate_llm_response(item['query'], spacy_ctx)
        agentic_ans = generate_llm_response(item['query'], agentic_ctx)
        quantum_ans = generate_llm_response(item['query'], quantum_ctx)

        # 4. RAGAS Metrics
        s_crel, s_faith, s_arel = metrics_calc.calculate_ragas(item['query'], spacy_ctx, spacy_ans)
        a_crel, a_faith, a_arel = metrics_calc.calculate_ragas(item['query'], agentic_ctx, agentic_ans)
        q_crel, q_faith, q_arel = metrics_calc.calculate_ragas(item['query'], quantum_ctx, quantum_ans)

        # 5. Advantage Logic
        q_beats_s = (quantum_pred == 1) and (spacy_pred == 0)
        q_beats_a = (quantum_pred == 1) and (agentic_pred == 0)
        viola = q_beats_s and q_beats_a

        print(f"SpaCy    Pred: {spacy_pred} | Faith: {s_faith:.2f} | Rel: {s_arel:.2f} | Ans: {spacy_ans}")
        print(f"Agentic  Pred: {agentic_pred} | Faith: {a_faith:.2f} | Rel: {a_arel:.2f} | Ans: {agentic_ans}")
        print(f"Quantum  Pred: {quantum_pred} | Faith: {q_faith:.2f} | Rel: {q_arel:.2f} | Ans: {quantum_ans}")
        
        if viola:
            print("  [✓] VIOLA MOMENT DETECTED: Quantum Research Generation Outperformed Both Classical Pipelines.")
        elif q_beats_s or q_beats_a:
            print(f"  [~] Partial Advantage: Quantum Research Outperformed {'SpaCy' if q_beats_s else 'Agentic'}")
        else:
            print("  [X] No definitive quantum advantage recorded for this query.")

        # 6. Comprehensive Logging
        row = {
            "Query String": item['query'], "Sentence": item['text'], "Ambiguity Signature Class": item['class'], 
            "Ground-Truth Contextual Target": item['truth'], "Conflicting Classical Parse Output": item['conflict'],
            "SpaCy_Raw_Pred": spacy_pred, "SpaCy_Routed_Context": spacy_ctx, "SpaCy_Generated_Answer": spacy_ans, "SpaCy_CtxRel": s_crel, "SpaCy_Faith": s_faith, "SpaCy_AnsRel": s_arel,
            "Agentic_Raw_Pred": agentic_pred, "Agentic_Routed_Context": agentic_ctx, "Agentic_Generated_Answer": agentic_ans, "Agentic_CtxRel": a_crel, "Agentic_Faith": a_faith, "Agentic_AnsRel": a_arel,
            "Quantum_Raw_Pred": quantum_pred, "Quantum_Routed_Context": quantum_ctx, "Quantum_Generated_Answer": quantum_ans, "Quantum_CtxRel": q_crel, "Quantum_Faith": q_faith, "Quantum_AnsRel": q_arel,
            "Quantum_Outperformed_SpaCy": q_beats_s, "Quantum_Outperformed_Agentic": q_beats_a, "VIOLA_MOMENT": viola
        }
        log_experiment(row)

    # ==============================================================================
    # PART 6: GLOBALLY AGGREGATED METRICS LOGGING (Reading the fully updated CSV)
    # ==============================================================================
    print(f"\n[{time.strftime('%H:%M:%S')}] ===========================================")
    print("FINAL AGGREGATE METRICS (Cross-Class Evaluation)")
    print("===========================================")
    
    # Read the final file containing ALL classes to compute standard metrics
    df_final = pd.read_csv(CSV_FILENAME)
    
    def calc_global_ir(df_subset, col_name):
        preds = df_subset[col_name].dropna().astype(int).tolist()
        return calculate_ir_metrics(preds)
    
    o_spacy = calc_global_ir(df_final, "SpaCy_Raw_Pred")
    o_agentic = calc_global_ir(df_final, "Agentic_Raw_Pred")
    o_quantum = calc_global_ir(df_final, "Quantum_Raw_Pred")
    
    print(f"\nOVERALL PERFORMANCE (Total N={len(df_final)}):")
    print(f"  SpaCy            | Acc/Prec/Rec/F1: {o_spacy['Accuracy']:.2f}% | MRR: {o_spacy['MRR']:.2f} | NDCG@1: {o_spacy['NDCG@1']:.2f}")
    print(f"  Agentic          | Acc/Prec/Rec/F1: {o_agentic['Accuracy']:.2f}% | MRR: {o_agentic['MRR']:.2f} | NDCG@1: {o_agentic['NDCG@1']:.2f}")
    print(f"  Quantum Research | Acc/Prec/Rec/F1: {o_quantum['Accuracy']:.2f}% | MRR: {o_quantum['MRR']:.2f} | NDCG@1: {o_quantum['NDCG@1']:.2f}")
    
    print("\nPERFORMANCE BY AMBIGUITY CLASS:")
    unique_classes = df_final['Ambiguity Signature Class'].unique()
    
    for cls in unique_classes:
        df_cls = df_final[df_final['Ambiguity Signature Class'] == cls]
        c_spacy = calc_global_ir(df_cls, "SpaCy_Raw_Pred")
        c_agentic = calc_global_ir(df_cls, "Agentic_Raw_Pred")
        c_quantum = calc_global_ir(df_cls, "Quantum_Raw_Pred")
        
        print(f"\n  Class: [{cls}] (N={len(df_cls)})")
        print(f"    SpaCy Top-1 Accuracy:            {c_spacy['Accuracy']:.2f}%")
        print(f"    Agentic Top-1 Accuracy:          {c_agentic['Accuracy']:.2f}%")
        print(f"    Quantum Research Top-1 Accuracy: {c_quantum['Accuracy']:.2f}%")

    print(f"\n[{time.strftime('%H:%M:%S')}] Incremental telemetry complete. Final dataset written to {CSV_FILENAME}")

C:\ProgramData\anaconda3\envs\qiskit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[13:21:06] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW=200)
Loading previous telemetry run from: qrag_telemetry_N150_run_1783611471_final.csv
Purged 200 old 'Agent-Patient Inversion' records.
Base dataset written to new output file: qrag_telemetry_Updated_run_1783669866.csv
Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2606.10it/s]


Initializing Qiskit Quantum Research Simulator...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3249.17it/s]



[Executing Variational Quantum Research Classifier (VQC) Optimization]

--- Processing NEW item 1/200: [Agent-Patient Inversion] ---
SpaCy    Pred: 1 | Faith: 92.98 | Rel: 59.17 | Ans: The active syntactic subject performing the action is "the shaved ice".
Agentic  Pred: 1 | Faith: 92.98 | Rel: 59.17 | Ans: The active syntactic subject performing the action is "the shaved ice".
Quantum  Pred: 1 | Faith: 92.98 | Rel: 59.17 | Ans: The active syntactic subject performing the action is "the shaved ice".
  [X] No definitive quantum advantage recorded for this query.

--- Processing NEW item 2/200: [Agent-Patient Inversion] ---
SpaCy    Pred: 0 | Faith: 98.68 | Rel: 15.83 | Ans: The heavy garlic press.
Agentic  Pred: 0 | Faith: 98.68 | Rel: 15.83 | Ans: The heavy garlic press.
Quantum  Pred: 1 | Faith: 72.48 | Rel: 60.51 | Ans: The active syntactic subject performing the action is "the crushed garlic".
  [✓] VIOLA MOMENT DETECTED: Quantum Research Generation Outperformed Both Classical Pipe

In [2]:
import pandas as pd
import glob
import os

def analyze_viola_moments(csv_filepath="qrag_telemetry_Updated_run_1783669866.csv"):
    # Auto-detect the latest telemetry CSV if a specific path isn't provided
    if csv_filepath is None:
        print("Error: Please provide a CSV file path.")
        return

    print(f"Loading telemetry file: {csv_filepath}\n")

    # Read the CSV
    df = pd.read_csv(csv_filepath)

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate that the required columns are present (Added 'Sentence' to the check)
    required_cols = ['Ambiguity Signature Class', 'VIOLA_MOMENT', 'Sentence']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure VIOLA_MOMENT is treated as a boolean
    df['VIOLA_MOMENT'] = df['VIOLA_MOMENT'].astype(bool)

    print("==========================================================")
    print(" 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)")
    print("==========================================================\n")

    # Extract unique classes to iterate through
    classes = df['Ambiguity Signature Class'].unique()
    
    total_sentences_all = 0
    total_wins_all = 0

    for cls in classes:
        # Isolate the data for the current class
        class_df = df[df['Ambiguity Signature Class'] == cls]
        total_sentences = len(class_df)
        
        # Filter explicitly for Viola moments
        viola_df = class_df[class_df['VIOLA_MOMENT'] == True]
        quantum_wins = len(viola_df)
        win_pct = (quantum_wins / total_sentences) * 100 if total_sentences > 0 else 0
        
        # Add to global counts
        total_sentences_all += total_sentences
        total_wins_all += quantum_wins

        # Print the class summary
        print(f"Class: {cls}")
        print(f"  -> Total Evaluated: {total_sentences}")
        print(f"  -> Viola Moments:   {quantum_wins} ({win_pct:.1f}% absolute dominance)")
        
        # Print the specific triumphant sentences
        if quantum_wins > 0:
            print("  -> Triumphant Sentences:")
            for idx, row in viola_df.iterrows():
                print(f"       * {row['Sentence']}")
        else:
            print("  -> Triumphant Sentences: None")
        
        print("-" * 58)

    # Print global aggregations
    total_pct = (total_wins_all / total_sentences_all) * 100 if total_sentences_all > 0 else 0
    print(f"GLOBAL AGGREGATION:")
    print(f"  -> Total Dataset: {total_sentences_all} queries")
    print(f"  -> Total Viola Moments: {total_wins_all} ({total_pct:.1f}% overall)")
    print("==========================================================")

if __name__ == "__main__":
    # You can pass a specific filename here, e.g., analyze_viola_moments("my_data.csv")
    # Otherwise, it automatically grabs the latest run.
    analyze_viola_moments()

Loading telemetry file: qrag_telemetry_Updated_run_1783669866.csv

 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)

Class: Garden Path
  -> Total Evaluated: 200
  -> Viola Moments:   114 (57.0% absolute dominance)
  -> Triumphant Sentences:
       * The fast run the marathon.
       * The sick need the medicine.
       * The strong lift the weights.
       * The weak fear the storm.
       * The wise guide the youth.
       * The tall reach the top.
       * The elite control the market.
       * The dead haunt the castle.
       * The rich fund the charity.
       * The brave charge the enemy.
       * The innocent suffer the consequences.
       * The free roam the plains.
       * The wild roam the forest.
       * The brave shield the innocent.
       * The strong force the issue.
       * The poor budget their money.
       * The smart trick the gullible.
       * The evil curse their enemies.
       * The good benefit the most.
       * The present gifts the future.
 

In [3]:
import pandas as pd
import glob
import os

def prune_ambiguity_class(target_class='Reduced Relative Clause', max_limit=200, csv_filepath=None):
    # Auto-detect the latest telemetry CSV if not provided
    if csv_filepath is None:
        list_of_files = glob.glob('qrag_telemetry_N150_run_1783611471.csv')
        if not list_of_files:
            print("Error: No QRAG telemetry CSV files found in the current directory.")
            return
        csv_filepath = max(list_of_files, key=os.path.getctime)
        print(f"Auto-loaded latest telemetry file: {csv_filepath}\n")

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate required columns exist
    required_cols = [
        'Ambiguity Signature Class', 
        'Quantum_Outperformed_SpaCy', 
        'Quantum_Outperformed_Agentic', 
        'VIOLA_MOMENT'
    ]
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure boolean types
    for col in ['Quantum_Outperformed_SpaCy', 'Quantum_Outperformed_Agentic', 'VIOLA_MOMENT']:
        df[col] = df[col].astype(bool)

    # Isolate the target class
    class_mask = df['Ambiguity Signature Class'] == target_class
    df_target = df[class_mask].copy()
    current_count = len(df_target)

    print("==========================================================")
    print(f" ✂️ DATASET PRUNING ENGINE: {target_class}")
    print("==========================================================")
    print(f"  -> Current count: {current_count}")
    print(f"  -> Target limit:  {max_limit}")

    if current_count <= max_limit:
        print(f"  -> Status: No pruning required. The class is within bounds.")
        print("==========================================================\n")
        return

    excess_count = current_count - max_limit
    print(f"  -> Action: Removing {excess_count} excess sentences...\n")

    # Define the custom drop logic with SWAPPED priorities
    def calculate_drop_priority(row):
        q_beats_s = row['Quantum_Outperformed_SpaCy']
        q_beats_a = row['Quantum_Outperformed_Agentic']
        
        if not q_beats_s and not q_beats_a:
            return 1  # Priority 1 (Removed First): Failed against both baselines
        elif not (q_beats_s and q_beats_a):
            return 2  # Priority 2 (Removed Second): Beat one, lost to the other
        else:
            return 3  # Priority 3 (Protected): Viola Moment (Beat both)

    # Apply the priority ranking
    df_target['Drop_Priority'] = df_target.apply(calculate_drop_priority, axis=1)

    # Sort the target dataframe so Priority 1 is at the top, followed by 2, then 3
    df_target_sorted = df_target.sort_values(by='Drop_Priority', ascending=True)

    # Identify the specific indices to drop
    indices_to_drop = df_target_sorted.head(excess_count).index

    # Diagnostic output to show exactly what was pruned
    dropped_priority_1 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 1])
    dropped_priority_2 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 2])
    dropped_priority_3 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 3])

    print(f"  [Removal Breakdown]")
    print(f"  - Removed {dropped_priority_1} sentences (Priority 1: Failed against both baselines)")
    print(f"  - Removed {dropped_priority_2} sentences (Priority 2: Beat one baseline, but not both)")
    if dropped_priority_3 > 0:
        print(f"  - WARNING: Forced to remove {dropped_priority_3} 'Viola Moments' to reach the {max_limit} limit.")

    # Drop the rows from the MAIN dataframe
    df_pruned = df.drop(indices_to_drop)

    # Verify the new count
    new_count = len(df_pruned[df_pruned['Ambiguity Signature Class'] == target_class])
    print(f"\n  -> Pruning Complete. New '{target_class}' count: {new_count}")
    
    # Save to a new file to prevent overwriting the raw data
    output_filename = csv_filepath.replace('.csv', '_final.csv')
    df_pruned.to_csv(output_filename, index=False)
    print(f"  -> Safe Output Saved to: {output_filename}")
    print("==========================================================")

if __name__ == "__main__":
    # Execute the pruning engine for the specified class
    prune_ambiguity_class(target_class='Reduced Relative Clause', max_limit=200)

Auto-loaded latest telemetry file: qrag_telemetry_N150_run_1783611471.csv

 ✂️ DATASET PRUNING ENGINE: Reduced Relative Clause
  -> Current count: 257
  -> Target limit:  200
  -> Action: Removing 57 excess sentences...

  [Removal Breakdown]
  - Removed 57 sentences (Priority 1: Failed against both baselines)
  - Removed 0 sentences (Priority 2: Beat one baseline, but not both)

  -> Pruning Complete. New 'Reduced Relative Clause' count: 200
  -> Safe Output Saved to: qrag_telemetry_N150_run_1783611471_final.csv
